In [30]:
import pandas as pd

path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\03_advanced_prep\lc_after_03_advanced_prep_basic+test_20260108_1516.csv"

df = pd.read_csv(path, low_memory=False)

print("Loaded:", df.shape)
df.head()

Loaded: (221428, 108)


,percent_bc_gt_75,num_tl_op_past_12m,zip_code,revol_bal,last_fico_range_high,sub_grade,emp_title,last_fico_range_low,total_rev_hi_lim,term,...,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Source Verified,verification_status_Verified
0,NaN,NaN,860xx,9.521422,749.0,B2,NaN,745.0,NaN,36,...,0,0,0,0,0,0,0,1,0,1
1,NaN,NaN,309xx,7.431300,499.0,C4,Ryder,0.0,NaN,60,...,0,0,0,0,0,0,0,1,1,0
2,NaN,NaN,606xx,7.991931,739.0,C5,NaN,735.0,NaN,36,...,0,1,0,0,0,0,0,1,0,0
3,NaN,NaN,917xx,8.630343,604.0,C1,AIR RESOURCES BOARD,600.0,NaN,36,...,0,0,0,0,0,0,0,1,1,0
4,NaN,NaN,972xx,10.232216,684.0,B5,University Medical Group,680.0,NaN,60,...,0,0,0,0,0,0,0,1,1,0


## הגדרת HAS TEST

In [31]:
# 1) דגל "יש טקסט" ב-desc
df['has_desc'] = df['desc'].notna() & df['desc'].astype(str).str.strip().ne('')

# 2) ספירה
df['has_desc'].value_counts(dropna=False)

has_desc
False    119642
True     101786
Name: count, dtype: int64

In [32]:
# 3) סינון לעבודה רק עם טקסט
df_txt = df.loc[df['has_desc']].copy()

print("All rows:", df.shape)
print("With desc:", df_txt.shape)

All rows: (221428, 109)
With desc: (101786, 109)


In [33]:
pd.Series({
    "default_rate_all": df['is_default'].mean(),
    "default_rate_with_desc": df_txt['is_default'].mean(),
    "n_all": len(df),
    "n_with_desc": len(df_txt)
}).round(4)

default_rate_all               0.1556
default_rate_with_desc         0.1536
n_all                     221428.0000
n_with_desc               101786.0000
dtype: float64

In [34]:
# כמה תצפיות עם desc בכל חודש
df_txt.groupby('issue_ym').size().describe()

df_txt.groupby('issue_ym').size().sort_values().head(10)

issue_ym
2010-01    475
2010-02    478
2010-03    503
2010-04    569
2010-08    619
2010-09    634
2010-05    653
2010-06    678
2010-07    715
2010-10    726
dtype: int64

## ניקוי טקסט

In [35]:
# הצגה נוחה של כמה תצפיות עם desc (כולל אורך טקסט)
cols = ['issue_ym', 'is_default', 'zip3', 'desc']
if 'desc_clean' in df_txt.columns:
    cols.append('desc_clean')

sample = (
    df_txt[cols]
    .assign(desc_len=df_txt['desc'].astype(str).str.len())
    .sort_values('desc_len', ascending=False)
    .head(20)
)

sample

,issue_ym,is_default,zip3,desc,desc_len
29733,2010-05,0,840,Borrower added on 05/18/10 > All funds from ...,3988
30691,2010-04,1,917,Borrower added on 04/16/10 > Hello Investors...,3986
25695,2010-09,0,629,Borrower added on 09/15/10 > The funds are g...,3976
20713,2011-01,0,83,Borrower added on 01/16/11 > I am using this...,3973
30467,2010-05,0,908,Borrower added on 04/27/10 > I am requesting...,3963
19870,2011-02,0,24,Borrower added on 02/07/11 > I am an IT Proj...,3960
30767,2010-04,0,600,Borrower added on 04/14/10 > My wife had a s...,3952
23624,2010-11,0,282,Borrower added on 11/08/10 > Help me go from...,3949
23512,2010-11,0,941,Borrower added on 11/11/10 > To my future in...,3919
8949,2011-08,1,921,I will use the funds to pay credit cards that...,3895


Cleans and standardizes raw loan descriptions

In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# --- 1) stopwords:
custom_stop = {
    "added","borrower","loan","lending","club",
    "thanks","thank","quot","consideration","time",
    "just","like","want","make","need", "br"
}
stopwords = set(ENGLISH_STOP_WORDS).union(custom_stop)

# --- 2) Lemmatizer (מנסה spaCy; אם אין, עובר ל-NLTK)
lemmatize_mode = None
nlp = None
wnl = None

try:
    import spacy
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
    lemmatize_mode = "spacy"
except Exception:
    try:
        import nltk
        from nltk.stem import WordNetLemmatizer
        wnl = WordNetLemmatizer()
        lemmatize_mode = "nltk"
    except Exception:
        lemmatize_mode = "none"

# --- 3) regex להסרת הפתיח: "Borrower added on 05/18/10 >""
prefix_re = re.compile(r"^\s*borrower\s+added\s+on\s+\d{2}/\d{2}/\d{2}\s*>\s*", re.IGNORECASE)

def clean_desc(text: str) -> str:
    if pd.isna(text):
        return ""

    s = str(text).lower()

    # remove prefix
    s = prefix_re.sub("", s)

    # remove numbers
    s = re.sub(r"\d+", " ", s)

    # remove punctuation / keep only letters + spaces
    s = re.sub(r"[^a-z\s]", " ", s)

    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()
    if not s:
        return ""

    tokens = s.split()

    # lemmatize
    if lemmatize_mode == "spacy":
        doc = nlp(" ".join(tokens))
        tokens = [t.lemma_ for t in doc]
    elif lemmatize_mode == "nltk":
        tokens = [wnl.lemmatize(t) for t in tokens]
    # else: בלי למטיזציה (אם אין ספרייה זמינה)

    # remove stopwords + tokens קצרים
    tokens = [t for t in tokens if t not in stopwords and len(t) >= 2]

    return " ".join(tokens)

# --- 4) הפעלה
df_txt["desc_clean"] = df_txt["desc"].apply(clean_desc)

# תצוגה מהירה
df_txt[["desc", "desc_clean"]].sample(10, random_state=42)

,desc,desc_clean
158198,Borrower added on 01/31/13 > Credit card rat...,credit card rate tripled consolidate high cred...
157919,Borrower added on 01/28/13 > I'm looking to ...,looking pay current credit card debit debit
79197,Borrower added on 09/16/13 > Paying off debt...,paying debt
46866,Borrower added on 11/26/13 > I have credit c...,credit cards rate rate reduce luck payments
88112,Borrower added on 08/25/13 > Lower interest ...,lower rate
90317,Borrower added on 08/19/13 > Debt consolidat...,debt consolidation
124062,Borrower added on 05/24/13 > I have three cr...,credit cards consolidate order debt free sched...
60117,Borrower added on 10/29/13 > I will use this...,use pay credit cards years payoff
148856,Borrower added on 03/05/13 > Just looking to...,looking pay high apr credit cards
173327,Borrower added on 11/27/12 > I have a couple...,couple credit cards consolidate payment instea...


In [37]:
df_txt['desc_clean'].sample(20, random_state=42).to_list()

['credit card rate tripled consolidate high credit cards',
 'looking pay current credit card debit debit',
 'paying debt',
 'credit cards rate rate reduce luck payments',
 'lower rate',
 'debt consolidation',
 'credit cards consolidate order debt free scheduled graduate mba human resources august order consolidate credit cards payment order debt free payment scheduled end date clear goal set secure finacial future',
 'use pay credit cards years payoff',
 'looking pay high apr credit cards',
 'couple credit cards consolidate payment instead having come month',
 'consolidate credit card medical bills',
 'plan use pay credit debt puts good position start fresh starting business began building design firm year ago plan force remainder',
 'medical issues family run credit card debt debt issues resolved',
 'help adult son pay recent medical bills accident',
 'purpose consolidate lower rate credit card debt incurred result family emergency began end december continued january february meant a

In [ ]:
# כמות התצפיות עם טקסט נקי
(df_txt['desc_clean'].str.len() > 0).mean()

np.float64(0.9983101801819504)

Cleans and standardizes raw loan descriptions on title and emp_title fileds

In [41]:
# stopwords כבר קיימים אצלך (ENGLISH_STOP_WORDS + custom_stop, כולל br)
# lemmatize_mode / nlp / wnl כבר קיימים אצלך מהניקוי של desc

def clean_short_text(text: str) -> str:
    if pd.isna(text):
        return ""
    s = str(text).lower()

    # numbers -> space
    s = re.sub(r"\d+", " ", s)

    # punctuation -> space, keep only letters + spaces
    s = re.sub(r"[^a-z\s]", " ", s)

    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()
    if not s:
        return ""

    tokens = s.split()

    # lemmatize (reuse your existing setup)
    if lemmatize_mode == "spacy":
        doc = nlp(" ".join(tokens))
        tokens = [t.lemma_ for t in doc]
    elif lemmatize_mode == "nltk":
        tokens = [wnl.lemmatize(t) for t in tokens]

    # remove stopwords + very short tokens
    tokens = [t for t in tokens if t not in stopwords and len(t) >= 2]

    return " ".join(tokens)

# apply to title + emp_title
df_txt["title_clean"] = df_txt["title"].apply(clean_short_text)
df_txt["emp_title_clean"] = df_txt["emp_title"].apply(clean_short_text)

# quick view
df_txt[["title","title_clean","emp_title","emp_title_clean"]].sample(10, random_state=42)

,title,title_clean,emp_title,emp_title_clean
158198,Thanks,,Pepsico,pepsico
157919,Debt consolidation,debt consolidation,Quirk ford,quirk ford
79197,Credit payment,credit payment,Paul Davis Restoration,paul davis restoration
46866,credit card,credit card,Tech,tech
88112,Freedom,freedom,Metropolitan Hospital,metropolitan hospital
90317,loan # 2,,Mt Carmel East Hospital,mt carmel east hospital
124062,Debt consolidation,debt consolidation,Tyson Foods,tyson foods
60117,Credit Card Payoff,credit card payoff,Police Officer,police officer
148856,Credit card refinancing,credit card refinancing,Architect Mechanical,architect mechanical
173327,Credit Card Refinance Needed,credit card refinance needed,Greg Jensen Originals,greg jensen originals


## הכנת פיצ'רים לעמודות טקסט

Bag-of-Words על ביגרמים, אורך טקסט, מספר מילים ייחודיות, אורך מילה ממוצע, כמות מילים שהוסרו בניקוי, ומדד גיוון (Type-Token Ratio)

In [ ]:

CLEAN_COLS = ["desc_clean", "title_clean", "emp_title_clean"]
RAW_COLS   = ["desc", "title", "emp_title"]

def _wc(s):  # word count
    return s.str.split().str.len()

def add_text_numeric_features(df):
    df = df.copy()

    for raw_c, clean_c in zip(RAW_COLS, CLEAN_COLS):
        raw = df[raw_c].fillna("").astype(str).str.strip()
        clean = df[clean_c].fillna("").astype(str).str.strip()

        raw_wc = _wc(raw.str.lower())
        clean_wc = _wc(clean)

        # אורך טקסט (תווים)
        df[f"{clean_c}_char_len"] = clean.str.len()

        # מס' מילים ייחודיות
        df[f"{clean_c}_uniq_words"] = clean.apply(lambda x: len(set(x.split())) if x else 0)

        # אורך מילה ממוצע
        df[f"{clean_c}_avg_word_len"] = clean.apply(lambda x: np.mean([len(w) for w in x.split()]) if x else 0.0)

        # כמות מילים שהוסרו (בקירוב שמרני)
        df[f"{clean_c}_removed_words"] = (raw_wc - clean_wc).clip(lower=0)

        # מדד גיוון (TTR)
        df[f"{clean_c}_ttr"] = np.where(
            clean_wc > 0,
            df[f"{clean_c}_uniq_words"] / clean_wc,
            0.0
        )

    return df

df_txt = add_text_numeric_features(df_txt)

# רשימת הפיצ'רים שנוצרו
num_feat_cols = [c for c in df_txt.columns if c.endswith(("_char_len","_uniq_words","_avg_word_len","_removed_words","_ttr"))]
print("Numeric text features:", len(num_feat_cols))
df_txt[num_feat_cols].describe().T.head(12)

Numeric text features: 15


,count,mean,std,min,25%,50%,75%,max
desc_clean_char_len,101786.0,102.851473,126.145524,0.0,35.000000,69.0,127.0,2312.0
desc_clean_uniq_words,101786.0,12.918712,13.259474,0.0,5.000000,9.0,17.0,211.0
desc_clean_avg_word_len,101786.0,6.006113,1.143028,0.0,5.333333,5.9,6.5,31.0
desc_clean_removed_words,101786.0,27.614367,29.138915,0.0,11.000000,20.0,35.0,560.0
desc_clean_ttr,101786.0,0.921810,0.123918,0.0,0.880000,1.0,1.0,1.0
title_clean_char_len,101786.0,14.423840,7.041214,0.0,9.000000,16.0,18.0,73.0
title_clean_uniq_words,101786.0,1.929008,0.933381,0.0,1.000000,2.0,2.0,10.0
title_clean_avg_word_len,101786.0,7.010203,2.794882,0.0,5.000000,7.0,8.5,40.0
title_clean_removed_words,101786.0,0.494115,0.837772,0.0,0.000000,0.0,1.0,11.0
title_clean_ttr,101786.0,0.963982,0.184884,0.0,1.000000,1.0,1.0,1.0


Transforms cleaned text into a numerical matrix using TF-IDF

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),     # אם אתה רוצה רק ביגרמים: (2,2)
    min_df=20,
    max_features=50000,
    sublinear_tf=True,
    norm="l2"
)

X_tfidf = tfidf.fit_transform(df_txt["desc_clean"].fillna("").astype(str))

print("X_tfidf shape:", X_tfidf.shape)
print("Example features:", tfidf.get_feature_names_out()[:20])

X_tfidf shape: (101786, 10129)
Example features: ['ability' 'ability pay' 'ability repay' 'able' 'able afford' 'able ahead'
 'able budget' 'able buy' 'able close' 'able consolidate' 'able credit'
 'able debt' 'able focus' 'able handle' 'able help' 'able invest'
 'able live' 'able lower' 'able manage' 'able meet']


In [45]:
df_txt["text_all_clean"] = (
    df_txt["desc_clean"].fillna("").astype(str) + " " +
    df_txt["title_clean"].fillna("").astype(str) + " " +
    df_txt["emp_title_clean"].fillna("").astype(str)
).str.strip()

tfidf_all = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=20,
    max_features=50000,
    sublinear_tf=True,
    norm="l2"
)

X_tfidf_all = tfidf_all.fit_transform(df_txt["text_all_clean"])

print("X_tfidf_all shape:", X_tfidf_all.shape)

X_tfidf_all shape: (101786, 12932)


In [46]:
from scipy.sparse import csr_matrix, hstack

text_num_cols = [c for c in df_txt.columns if c.endswith(("_char_len","_uniq_words","_avg_word_len","_removed_words","_ttr"))]
X_text_num = csr_matrix(df_txt[text_num_cols].values)

X_text_all = hstack([X_text_num, X_tfidf_all]).tocsr()
print("X_text_all shape:", X_text_all.shape)

X_text_all shape: (101786, 12947)


## הרצת מודלים

XGBOOST

Random split

In [47]:
from sklearn.model_selection import train_test_split

y = df_txt["is_default"].astype(int).values

idx = np.arange(len(df_txt))
idx_tr, idx_te = train_test_split(
    idx,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Train size:", len(idx_tr), "Test size:", len(idx_te))
print("Default rate train:", y[idx_tr].mean().round(4))
print("Default rate test :", y[idx_te].mean().round(4))

Train size: 71250 Test size: 30536
Default rate train: 0.1536
Default rate test : 0.1536


In [ ]:
from scipy.sparse import csr_matrix, hstack
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# y
y = df_txt["is_default"].astype(int).values
y_tr = y[idx_tr]
y_te = y[idx_te]

# imbalance weight
pos = y_tr.sum()
neg = len(y_tr) - pos
spw = neg / max(pos, 1)

# 1) STRUCTURED
raw_text_cols = [
    "desc","title","emp_title",
    "desc_clean","title_clean","emp_title_clean",
    "text_all_clean","has_desc"
]
drop_always = set(["is_default"]) | set(raw_text_cols)

structured_cols = [
    c for c in df_txt.columns
    if c not in drop_always and np.issubdtype(df_txt[c].dtype, np.number)
]

X_struct = csr_matrix(df_txt[structured_cols].values)
X_struct_tr = X_struct[idx_tr]
X_struct_te = X_struct[idx_te]

def fit_eval_xgb(Xtr, Xte, ytr, yte, label):
    model = XGBClassifier(
        n_estimators=1200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        min_child_weight=1,
        gamma=0,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        n_jobs=-1,
        scale_pos_weight=spw,
        random_state=242
    )
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(yte, p)
    pr  = average_precision_score(yte, p)
    print(f"{label}  AUC={auc:.4f} | PR-AUC={pr:.4f}")
    return auc, pr

auc1, pr1 = fit_eval_xgb(X_struct_tr, X_struct_te, y_tr, y_te, "XGB: STRUCTURED ONLY")

# 2) STRUCTURED + TEXT FEATURES
# X_text_all כבר אצלך: (numeric text + TF-IDF)
X_full = hstack([X_struct, X_text_all]).tocsr()
X_full_tr = X_full[idx_tr]
X_full_te = X_full[idx_te]

auc2, pr2 = fit_eval_xgb(X_full_tr, X_full_te, y_tr, y_te, "XGB: STRUCTURED + TEXT (NUM+TFIDF)")

print(f"ΔAUC={auc2-auc1:+.4f} | ΔPR-AUC={pr2-pr1:+.4f}")

XGB: STRUCTURED ONLY  AUC=0.9043 | PR-AUC=0.6042
XGB: STRUCTURED + TEXT (NUM+TFIDF)  AUC=0.9032 | PR-AUC=0.5951
ΔAUC=-0.0012 | ΔPR-AUC=-0.0091


LOGIT

In [ ]:
from scipy.sparse import csr_matrix, hstack
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

# y
y = df_txt["is_default"].astype(int).values
y_tr = y[idx_tr]
y_te = y[idx_te]

# -------- 1) STRUCTURED ONLY (dense, עם אימפיוט + סטנדרטיזציה) --------
raw_text_cols = [
    "desc","title","emp_title",
    "desc_clean","title_clean","emp_title_clean",
    "text_all_clean","has_desc"
]
drop_always = set(["is_default"]) | set(raw_text_cols)

structured_cols = [
    c for c in df_txt.columns
    if c not in drop_always and np.issubdtype(df_txt[c].dtype, np.number)
]

X_struct = df_txt[structured_cols].to_numpy()

pipe_struct = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        solver="lbfgs",
        penalty="l2",
        C=1.0,
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1,
        random_state=242
    ))
])

pipe_struct.fit(X_struct[idx_tr], y_tr)
p0 = pipe_struct.predict_proba(X_struct[idx_te])[:, 1]
auc0 = roc_auc_score(y_te, p0)
pr0  = average_precision_score(y_te, p0)
print(f"LR: STRUCTURED ONLY  AUC={auc0:.4f} | PR-AUC={pr0:.4f}")

# 2) STRUCTURED + TEXT (TF-IDF + text numeric)
# text numeric cols (15)
text_num_cols = [c for c in df_txt.columns if c.endswith(("_char_len","_uniq_words","_avg_word_len","_removed_words","_ttr"))]

# אימפיוט ל-structured ול-text numeric ואז hstack עם TF-IDF
imp_s = SimpleImputer(strategy="median")
X_struct_imp = imp_s.fit_transform(df_txt[structured_cols])

imp_t = SimpleImputer(strategy="median")
X_text_imp = imp_t.fit_transform(df_txt[text_num_cols])

X_full = hstack([csr_matrix(X_struct_imp), csr_matrix(X_text_imp), X_tfidf_all]).tocsr()

lr_sparse = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=1.0,
    max_iter=800,
    tol=1e-3,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

lr_sparse.fit(X_full[idx_tr], y_tr)
p1 = lr_sparse.predict_proba(X_full[idx_te])[:, 1]
auc1 = roc_auc_score(y_te, p1)
pr1  = average_precision_score(y_te, p1)
print(f"LR: STRUCTURED + TEXT (NUM+TFIDF)  AUC={auc1:.4f} | PR-AUC={pr1:.4f}")

print(f"ΔAUC={auc1-auc0:+.4f} | ΔPR-AUC={pr1-pr0:+.4f}")

c:\Users\ariel\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\ariel\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


LR: STRUCTURED ONLY  AUC=0.8978 | PR-AUC=0.5682


c:\Users\ariel\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\ariel\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


LR: STRUCTURED + TEXT (NUM+TFIDF)  AUC=0.8942 | PR-AUC=0.5600
ΔAUC=-0.0036 | ΔPR-AUC=-0.0082


## הוגנות לפי ZIP3

In [ ]:

# ========= inputs from your XGBoost runs =========
# y_te already exists (from your split)
# IMPORTANT: you must have the probability vectors from the two XGBoost models:
# p_struct = model_struct.predict_proba(X_struct_te)[:,1]
# p_full   = model_full.predict_proba(X_full_te)[:,1]
#
# If you didn't save them, re-run XGBoost and store them as p_struct / p_full.

zip_te = df_txt.iloc[idx_te]["zip3"].astype(str).values  # <-- use iloc (fixes KeyError)

def fairness_gaps_by_group(y_true, p, groups, threshold=0.7, min_group_n=200, min_group_pos=20):
    """
    Computes group-wise rates on TEST and returns:
    - FNR_gap: max(FNR) - min(FNR)
    - FPR_gap: max(FPR) - min(FPR)
    - EO_gap : max( max(TPR)-min(TPR), max(FPR)-min(FPR) )
    Only groups that satisfy size filters are included.
    """
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p)
    yhat = (p >= threshold).astype(int)

    df = pd.DataFrame({"y": y_true, "yhat": yhat, "g": groups})

    # filter small groups (stability)
    grp = df.groupby("g").agg(n=("y", "size"), pos=("y", "sum"))
    keep = grp[(grp["n"] >= min_group_n) & (grp["pos"] >= min_group_pos)].index
    df = df[df["g"].isin(keep)]

    def rates(sub):
        y = sub["y"].to_numpy()
        yh = sub["yhat"].to_numpy()
        tp = ((yh == 1) & (y == 1)).sum()
        tn = ((yh == 0) & (y == 0)).sum()
        fp = ((yh == 1) & (y == 0)).sum()
        fn = ((yh == 0) & (y == 1)).sum()

        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan

        return pd.Series({"TPR": tpr, "FPR": fpr, "FNR": fnr, "n": len(y), "pos": y.sum()})

    per = df.groupby("g").apply(rates).dropna()

    # gaps (max-min across groups)
    fnr_gap = per["FNR"].max() - per["FNR"].min()
    fpr_gap = per["FPR"].max() - per["FPR"].min()
    tpr_gap = per["TPR"].max() - per["TPR"].min()
    eo_gap  = max(fpr_gap, tpr_gap)

    out = {
        "threshold": threshold,
        "n_groups_used": int(per.shape[0]),
        "FNR_gap": float(fnr_gap),
        "FPR_gap": float(fpr_gap),
        "EO_gap": float(eo_gap),
        # optional: table if you want to inspect top groups
        "per_group": per.sort_values("n", ascending=False)
    }
    return out

# ====== choose threshold ======
thr = 0.7  # change if you want business threshold

# ====== run for BOTH XGBoost models ======
res_struct = fairness_gaps_by_group(y_te, p_struct, zip_te, threshold=thr, min_group_n=200, min_group_pos=20)
res_full   = fairness_gaps_by_group(y_te, p_full,   zip_te, threshold=thr, min_group_n=200, min_group_pos=20)

print(f"XGB STRUCTURED ONLY (groups={res_struct['n_groups_used']}, thr={thr}) "
      f"FNR_gap={res_struct['FNR_gap']:.4f} | FPR_gap={res_struct['FPR_gap']:.4f} | EO_gap={res_struct['EO_gap']:.4f}")

print(f"XGB STRUCTURED + TEXT (groups={res_full['n_groups_used']}, thr={thr}) "
      f"FNR_gap={res_full['FNR_gap']:.4f} | FPR_gap={res_full['FPR_gap']:.4f} | EO_gap={res_full['EO_gap']:.4f}")

print("Δ(TEXT-STRUCT): "
      f"FNR_gap={res_full['FNR_gap']-res_struct['FNR_gap']:+.4f} | "
      f"FPR_gap={res_full['FPR_gap']-res_struct['FPR_gap']:+.4f} | "
      f"EO_gap={res_full['EO_gap']-res_struct['EO_gap']:+.4f}")

# Optional: inspect group tables
# display(res_struct["per_group"].head(10))
# display(res_full["per_group"].head(10))

XGB STRUCTURED ONLY (groups=18, thr=0.7) FNR_gap=0.2601 | FPR_gap=0.0974 | EO_gap=0.2601
XGB STRUCTURED + TEXT (groups=18, thr=0.7) FNR_gap=0.2757 | FPR_gap=0.0994 | EO_gap=0.2757
Δ(TEXT-STRUCT): FNR_gap=+0.0156 | FPR_gap=+0.0020 | EO_gap=+0.0156


C:\Users\ariel\AppData\Local\Temp\ipykernel_22816\2708113351.py:44: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per = df.groupby("g").apply(rates).dropna()
C:\Users\ariel\AppData\Local\Temp\ipykernel_22816\2708113351.py:44: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per = df.groupby("g").apply(rates).dropna()


logit

In [58]:

# =======================
# Inputs from your LOGIT run
# =======================
# y_te exists (from split)
# p0 = pipe_struct.predict_proba(X_struct[idx_te])[:,1]          # structured only
# p1 = lr_sparse.predict_proba(X_full[idx_te])[:,1]              # structured + text (NUM+TFIDF)

zip_te = df_txt.iloc[idx_te]["zip3"].astype(str).values  # critical: iloc (idx_te are positions)

def fairness_gaps_by_group(y_true, p, groups, threshold=0.7, min_group_n=200, min_group_pos=20):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p)
    yhat = (p >= threshold).astype(int)

    df = pd.DataFrame({"y": y_true, "yhat": yhat, "g": groups})

    # stability filter
    grp = df.groupby("g").agg(n=("y","size"), pos=("y","sum"))
    keep = grp[(grp["n"] >= min_group_n) & (grp["pos"] >= min_group_pos)].index
    df = df[df["g"].isin(keep)]

    # per-group confusion rates
    def rates(sub):
        y = sub["y"].to_numpy()
        yh = sub["yhat"].to_numpy()
        tp = ((yh == 1) & (y == 1)).sum()
        tn = ((yh == 0) & (y == 0)).sum()
        fp = ((yh == 1) & (y == 0)).sum()
        fn = ((yh == 0) & (y == 1)).sum()

        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        return pd.Series({"TPR": tpr, "FPR": fpr, "FNR": fnr, "n": len(y), "pos": y.sum()})

    per = df.groupby("g").apply(rates).dropna()

    fnr_gap = per["FNR"].max() - per["FNR"].min()
    fpr_gap = per["FPR"].max() - per["FPR"].min()
    tpr_gap = per["TPR"].max() - per["TPR"].min()
    eo_gap  = max(fpr_gap, tpr_gap)

    return {
        "threshold": threshold,
        "n_groups_used": int(per.shape[0]),
        "FNR_gap": float(fnr_gap),
        "FPR_gap": float(fpr_gap),
        "EO_gap": float(eo_gap),
        "per_group": per.sort_values("n", ascending=False)
    }

thr = 0.7

res_lr_struct = fairness_gaps_by_group(y_te, p0, zip_te, threshold=thr, min_group_n=200, min_group_pos=20)
res_lr_text   = fairness_gaps_by_group(y_te, p1, zip_te, threshold=thr, min_group_n=200, min_group_pos=20)

print(f"LR STRUCTURED ONLY (groups={res_lr_struct['n_groups_used']}, thr={thr}) "
      f"FNR_gap={res_lr_struct['FNR_gap']:.4f} | FPR_gap={res_lr_struct['FPR_gap']:.4f} | EO_gap={res_lr_struct['EO_gap']:.4f}")

print(f"LR STRUCTURED + TEXT (groups={res_lr_text['n_groups_used']}, thr={thr}) "
      f"FNR_gap={res_lr_text['FNR_gap']:.4f} | FPR_gap={res_lr_text['FPR_gap']:.4f} | EO_gap={res_lr_text['EO_gap']:.4f}")

print("Δ(TEXT-STRUCT): "
      f"FNR_gap={res_lr_text['FNR_gap']-res_lr_struct['FNR_gap']:+.4f} | "
      f"FPR_gap={res_lr_text['FPR_gap']-res_lr_struct['FPR_gap']:+.4f} | "
      f"EO_gap={res_lr_text['EO_gap']-res_lr_struct['EO_gap']:+.4f}")

# Optional: inspect top groups
# display(res_lr_struct["per_group"].head(10))
# display(res_lr_text["per_group"].head(10))

LR STRUCTURED ONLY (groups=18, thr=0.7) FNR_gap=0.3138 | FPR_gap=0.1057 | EO_gap=0.3138
LR STRUCTURED + TEXT (groups=18, thr=0.7) FNR_gap=0.2412 | FPR_gap=0.0833 | EO_gap=0.2412
Δ(TEXT-STRUCT): FNR_gap=-0.0726 | FPR_gap=-0.0224 | EO_gap=-0.0726


C:\Users\ariel\AppData\Local\Temp\ipykernel_22816\3077219683.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per = df.groupby("g").apply(rates).dropna()
C:\Users\ariel\AppData\Local\Temp\ipykernel_22816\3077219683.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per = df.groupby("g").apply(rates).dropna()


brier_gap

In [65]:

# zip3 של ה-TEST (חשוב iloc כי idx_te הם positions)
zip_te = df_txt.iloc[idx_te]["zip3"].astype(str).values

def brier_by_group(y_true, p, groups, min_group_n=200, min_group_pos=20):
    """
    Returns:
      - overall_brier
      - brier_gap (max-min across groups, after stability filter)
      - per_group table
      - n_groups_used
      - N used
    """
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p)

    df = pd.DataFrame({"y": y_true, "p": p, "g": groups})
    df["sq_err"] = (df["p"] - df["y"]) ** 2

    # stability filter (same idea as before)
    grp = df.groupby("g").agg(n=("y","size"), pos=("y","sum"))
    keep = grp[(grp["n"] >= min_group_n) & (grp["pos"] >= min_group_pos)].index
    df_f = df[df["g"].isin(keep)]

    per = df_f.groupby("g").agg(
        n=("y","size"),
        pos=("y","sum"),
        brier=("sq_err","mean")
    ).sort_values("n", ascending=False)

    overall_brier = float(df["sq_err"].mean())
    brier_gap = float(per["brier"].max() - per["brier"].min())

    return {
        "overall_brier": overall_brier,
        "brier_gap": brier_gap,
        "n_groups_used": int(per.shape[0]),
        "N_used_for_fairness": int(per["n"].sum()),
        "per_group": per
    }

# ===== run for both XGBoost models (NO retrain) =====
res_brier_struct = brier_by_group(y_te, p_struct, zip_te, min_group_n=200, min_group_pos=20)
res_brier_full   = brier_by_group(y_te, p_full,   zip_te, min_group_n=200, min_group_pos=20)

print(f"XGB STRUCTURED ONLY:    overall_brier={res_brier_struct['overall_brier']:.4f} | "
      f"brier_gap={res_brier_struct['brier_gap']:.4f} | groups={res_brier_struct['n_groups_used']} | N_used={res_brier_struct['N_used_for_fairness']}")

print(f"XGB STRUCTURED + TEXT:  overall_brier={res_brier_full['overall_brier']:.4f} | "
      f"brier_gap={res_brier_full['brier_gap']:.4f} | groups={res_brier_full['n_groups_used']} | N_used={res_brier_full['N_used_for_fairness']}")

print("Δ(TEXT-STRUCT): "
      f"overall_brier={res_brier_full['overall_brier']-res_brier_struct['overall_brier']:+.4f} | "
      f"brier_gap={res_brier_full['brier_gap']-res_brier_struct['brier_gap']:+.4f}")


# display(res_brier_struct["per_group"].head(10))
# display(res_brier_full["per_group"].head(10))
# display(res_brier_struct["per_group"].sort_values("brier").head(10))
# display(res_brier_struct["per_group"].sort_values("brier").tail(10))

XGB STRUCTURED ONLY:    overall_brier=0.1220 | brier_gap=0.0725 | groups=18 | N_used=4914
XGB STRUCTURED + TEXT:  overall_brier=0.1248 | brier_gap=0.0715 | groups=18 | N_used=4914
Δ(TEXT-STRUCT): overall_brier=+0.0028 | brier_gap=-0.0010


In [ ]:

zip_te = df_txt.iloc[idx_te]["zip3"].astype(str).values  # iloc חשוב

def brier_by_group(y_true, p, groups, min_group_n=200, min_group_pos=20):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p)

    df = pd.DataFrame({"y": y_true, "p": p, "g": groups})
    df["sq_err"] = (df["p"] - df["y"]) ** 2

    # stability filter
    grp = df.groupby("g").agg(n=("y","size"), pos=("y","sum"))
    keep = grp[(grp["n"] >= min_group_n) & (grp["pos"] >= min_group_pos)].index
    df_f = df[df["g"].isin(keep)]

    per = df_f.groupby("g").agg(
        n=("y","size"),
        pos=("y","sum"),
        brier=("sq_err","mean")
    ).sort_values("n", ascending=False)

    overall_brier = float(df["sq_err"].mean())
    brier_gap = float(per["brier"].max() - per["brier"].min())

    return {
        "overall_brier": overall_brier,
        "brier_gap": brier_gap,
        "n_groups_used": int(per.shape[0]),
        "N_used_for_fairness": int(per["n"].sum()),
        "per_group": per
    }

res_lr_struct = brier_by_group(y_te, p0, zip_te, min_group_n=200, min_group_pos=20)
res_lr_text   = brier_by_group(y_te, p1, zip_te, min_group_n=200, min_group_pos=20)

print(f"LR STRUCTURED ONLY:   overall_brier={res_lr_struct['overall_brier']:.4f} | "
      f"brier_gap={res_lr_struct['brier_gap']:.4f} | groups={res_lr_struct['n_groups_used']} | N_used={res_lr_struct['N_used_for_fairness']}")

print(f"LR STRUCTURED + TEXT: overall_brier={res_lr_text['overall_brier']:.4f} | "
      f"brier_gap={res_lr_text['brier_gap']:.4f} | groups={res_lr_text['n_groups_used']} | N_used={res_lr_text['N_used_for_fairness']}")

print("Δ(TEXT-STRUCT): "
      f"overall_brier={res_lr_text['overall_brier']-res_lr_struct['overall_brier']:+.4f} | "
      f"brier_gap={res_lr_text['brier_gap']-res_lr_struct['brier_gap']:+.4f}")

# display(res_lr_struct["per_group"].sort_values("brier").head(10))
# display(res_lr_struct["per_group"].sort_values("brier").tail(10))

LR STRUCTURED ONLY:   overall_brier=0.1335 | brier_gap=0.0765 | groups=18 | N_used=4914
LR STRUCTURED + TEXT: overall_brier=0.1360 | brier_gap=0.0660 | groups=18 | N_used=4914
Δ(TEXT-STRUCT): overall_brier=+0.0026 | brier_gap=-0.0105


A summary table comparing fairness and error metrics model variants to measure the impact of textual featues on group disparities

In [63]:

zip_te = df_txt.iloc[idx_te]["zip3"].astype(str).values

def fairness_gaps_by_group(y_true, p, groups, threshold=0.7, min_group_n=200, min_group_pos=20):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p)
    yhat = (p >= threshold).astype(int)

    df = pd.DataFrame({"y": y_true, "yhat": yhat, "g": groups})

    # stability filter
    grp = df.groupby("g").agg(n=("y","size"), pos=("y","sum"))
    keep = grp[(grp["n"] >= min_group_n) & (grp["pos"] >= min_group_pos)].index
    df = df[df["g"].isin(keep)]

    def rates(sub):
        y = sub["y"].to_numpy()
        yh = sub["yhat"].to_numpy()
        tp = ((yh==1) & (y==1)).sum()
        tn = ((yh==0) & (y==0)).sum()
        fp = ((yh==1) & (y==0)).sum()
        fn = ((yh==0) & (y==1)).sum()

        fpr = fp/(fp+tn) if (fp+tn)>0 else np.nan
        fnr = fn/(fn+tp) if (fn+tp)>0 else np.nan
        return pd.Series({"FPR": fpr, "FNR": fnr, "n": len(y), "pos": y.sum()})

    per = df.groupby("g").apply(rates).dropna()

    return {
        "n_groups_used": int(per.shape[0]),
        "N_used": int(per["n"].sum()),
        "FNR_gap": float(per["FNR"].max() - per["FNR"].min()),
        "FPR_gap": float(per["FPR"].max() - per["FPR"].min()),
    }

def brier_gap_by_group(y_true, p, groups, min_group_n=200, min_group_pos=20):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p)

    df = pd.DataFrame({"y": y_true, "p": p, "g": groups})
    df["sq_err"] = (df["p"] - df["y"]) ** 2

    grp = df.groupby("g").agg(n=("y","size"), pos=("y","sum"))
    keep = grp[(grp["n"] >= min_group_n) & (grp["pos"] >= min_group_pos)].index
    df = df[df["g"].isin(keep)]

    per = df.groupby("g").agg(n=("y","size"), brier=("sq_err","mean")).dropna()
    return {
        "n_groups_used": int(per.shape[0]),
        "N_used": int(per["n"].sum()),
        "Brier_gap": float(per["brier"].max() - per["brier"].min()),
    }

def pack_row(model_name, variant, y_true, p, groups, thr=0.5, min_n=200, min_pos=20):
    f = fairness_gaps_by_group(y_true, p, groups, threshold=thr, min_group_n=min_n, min_group_pos=min_pos)
    b = brier_gap_by_group(y_true, p, groups, min_group_n=min_n, min_group_pos=min_pos)

    # sanity: should be same groups/N used
    return {
        "Model": model_name,
        "Variant": variant,
        "threshold": thr,
        "groups_used": f["n_groups_used"],
        "N_used": f["N_used"],
        "FNR_gap": f["FNR_gap"],
        "FPR_gap": f["FPR_gap"],
        "Brier_gap": b["Brier_gap"],
    }

thr = 0.7
min_n = 200
min_pos = 20

rows = []

# --- XGBoost ---
rows.append(pack_row("XGBoost", "Structured",      y_te, p_struct, zip_te, thr, min_n, min_pos))
rows.append(pack_row("XGBoost", "Structured+Text", y_te, p_full,   zip_te, thr, min_n, min_pos))

# --- Logistic ---
rows.append(pack_row("Logistic", "Structured",      y_te, p0, zip_te, thr, min_n, min_pos))
rows.append(pack_row("Logistic", "Structured+Text", y_te, p1, zip_te, thr, min_n, min_pos))

df_metrics = pd.DataFrame(rows)

# add deltas per model (Text - Structured)
delta_rows = []
for m in df_metrics["Model"].unique():
    base = df_metrics[(df_metrics["Model"]==m) & (df_metrics["Variant"]=="Structured")].iloc[0]
    txt  = df_metrics[(df_metrics["Model"]==m) & (df_metrics["Variant"]=="Structured+Text")].iloc[0]
    delta_rows.append({
        "Model": m,
        "Variant": "Δ(Text-Struct)",
        "threshold": thr,
        "groups_used": txt["groups_used"],
        "N_used": txt["N_used"],
        "FNR_gap": txt["FNR_gap"] - base["FNR_gap"],
        "FPR_gap": txt["FPR_gap"] - base["FPR_gap"],
        "Brier_gap": txt["Brier_gap"] - base["Brier_gap"],
    })

df_metrics = pd.concat([df_metrics, pd.DataFrame(delta_rows)], ignore_index=True)

# formatting
df_metrics = df_metrics.sort_values(["Model","Variant"]).reset_index(drop=True)
df_metrics[["FNR_gap","FPR_gap","Brier_gap"]] = df_metrics[["FNR_gap","FPR_gap","Brier_gap"]].round(4)

df_metrics

C:\Users\ariel\AppData\Local\Temp\ipykernel_22816\619165128.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per = df.groupby("g").apply(rates).dropna()
C:\Users\ariel\AppData\Local\Temp\ipykernel_22816\619165128.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per = df.groupby("g").apply(rates).dropna()
C:\Users\ariel\AppData\Local\Temp\ipykernel_22816\619165128.py:27: FutureWarning: DataFrameGroupBy.apply o

,Model,Variant,threshold,groups_used,N_used,FNR_gap,FPR_gap,Brier_gap
0,Logistic,Structured,0.7,18,4914,0.3138,0.1057,0.0765
1,Logistic,Structured+Text,0.7,18,4914,0.2412,0.0833,0.0660
2,Logistic,Δ(Text-Struct),0.7,18,4914,-0.0726,-0.0224,-0.0105
3,XGBoost,Structured,0.7,18,4914,0.2601,0.0974,0.0725
4,XGBoost,Structured+Text,0.7,18,4914,0.2757,0.0994,0.0715
5,XGBoost,Δ(Text-Struct),0.7,18,4914,0.0156,0.0020,-0.0010
